# [ RENPLAY AI ] - PIPELINE OFICIAL NO GOOGLE COLAB
### Motor de Traducao, Analise Narrativa e Treinamento com Qwen3.5-4B (Unsloth FP16)
---
**Recursos Ativos:**
- Download direto via **Pixeldrain** (`https://pixeldrain.com/u/...`)
- Descompactacao de pacotes `.rpa` e descompilacao automatica de `.rpyc` para `.rpy`
- Relatorio de analise de personagens, rotas e variaveis criticas do jogo
- Fine-Tuning LoRA do **Qwen3.5-4B** no contexto do jogo
- Traducao em lote com barra de progresso em tempo real e protecao de tags
- **Entrega em Arquivo Unico:** Gera o patch drop-in com tela de idiomas integrada para a pasta `game/`


## 1. Verificacao de Hardware & GPU
Verifica se a GPU do Colab esta ativa (NVIDIA T4 16GB em modo FP16).

In [ ]:
!nvidia-smi
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Nenhuma'
print(f'[INFO] GPU Detectada: {gpu_name}')
if 'T4' in gpu_name:
    print('[OK] GPU Tesla T4 (Turing) pronta. Precisao configurada para torch.float16.')


## 2. Montar Google Drive & Clonar Repositorio
Monta o Google Drive para persistencia permanente de patches, memoria e checkpoints.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import sys

drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/RenplayAI')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / 'patches').mkdir(exist_ok=True)
(DRIVE_DIR / 'checkpoints').mkdir(exist_ok=True)
(DRIVE_DIR / 'data').mkdir(exist_ok=True)

REPO_URL = 'https://github.com/Ricardosvyt/ia-renpy.git'
LOCAL_REPO = Path('/content/renplay-ai')

if not (LOCAL_REPO / '.git').exists():
    print('[INFO] Clonando repositorio Renplay AI do GitHub...')
    !rm -rf /content/renplay-ai
    !git clone {REPO_URL} /content/renplay-ai
else:
    print('[INFO] Atualizando repositorio...')
    !cd /content/renplay-ai && git pull

if str(LOCAL_REPO) not in sys.path:
    sys.path.insert(0, str(LOCAL_REPO))
print('[OK] Repositorio Renplay AI sincronizado!')


## 3. Instalacao de Dependencias & Unsloth
Instala Unsloth, Unsloth Zoo, UnRPA e ferramentas de processamento.

In [ ]:
%%capture
!pip install unsloth_zoo
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install unrpa requests tqdm rich pydantic py7zr datasets
print('[OK] Todas as dependencias instaladas com sucesso!')


## 4. Gerenciamento da Fila de Jogos
Cole seu link do **Pixeldrain** e adicione o jogo a fila. Voce pode executar esta celula quantas vezes quiser para colocar mais jogos na fila.

In [ ]:
from core.queue_manager import QueueManager

queue_path = DRIVE_DIR / 'data' / 'fila_jogos.json'
queue_mgr = QueueManager(queue_file_path=queue_path)

# --- ADICIONAR LINK DO JOGO AQUI ---
LINK_PIXELDRAIN = 'https://pixeldrain.com/u/rhqrh5ij'
NOME_JOGO = 'AvertonMethod'

queue_mgr.add_game(url=LINK_PIXELDRAIN, name=NOME_JOGO, target_languages=['pt-BR'])

print('
[FILA ATUAL DE PROCESSAMENTO]')
for item in queue_mgr.list_queue():
    print(f"- [{item['status'].upper()}] {item['name']} ({item['url']})")


## 5. Carregar Modelo Qwen3.5-4B (FP16 Unsloth)
Inicializa o modelo base em precisao 16-bit com os aceleradores de atencao do Unsloth.

In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = 'unsloth/Qwen3.5-4B'
max_seq_length = 2048

print(f'[INFO] Carregando {model_name} em FP16...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    dtype=torch.float16,
)
print('[OK] Modelo Qwen3.5-4B pronto para analise, treino e traducao!')


## 6. Pipeline Completo: Download, Extracao, Analise, Treino & Patch Unico
Executa o fluxo completo do jogo da fila. Exibe o relatorio analitico, efetua o fine-tuning LoRA e gera o patch pronto no Google Drive.

In [ ]:
from core.pixeldrain_downloader import PixeldrainDownloader
from core.renpy_extractor import RenpyExtractor
from core.tag_protector import TagProtector
from core.sqlite_memory import SQLiteMemory
from core.patch_builder import PatchBuilder
from core.trainer import RenplayTrainer
from unsloth import FastLanguageModel
from tqdm import tqdm

memory = SQLiteMemory(db_path=DRIVE_DIR / 'data' / 'memoria_traducoes.sqlite')
protector = TagProtector()
downloader = PixeldrainDownloader()
patch_builder = PatchBuilder(output_dir=DRIVE_DIR / 'patches')

item = queue_mgr.get_next_pending()
if not item:
    print('[INFO] Nenhum jogo com status PENDING na fila. Verifique a Celula 4!')
else:
    print(f"\n[INICIANDO PIPELINE] {item['name']} ({item['url']})")
    queue_mgr.update_status(item['id'], 'downloading')
    
    try:
        # 1. Download do Pixeldrain
        print(f"\n=== ETAPA 1: DOWNLOAD DO PIXELDRAIN ===")
        archive = downloader.download(item['url'])
        
        # 2. Descompactacao do Arquivo Principal
        print(f"\n=== ETAPA 2: DESCOMPACTACAO DO JOGO ===")
        queue_mgr.update_status(item['id'], 'extracting')
        game_root = downloader.extract_game(archive)
        
        # 3. Descompilacao (.rpa e .rpyc) & Analise Narrativa do Jogo
        print(f"\n=== ETAPA 3: ANALISE NARRATIVA & DESCOMPILACAO ===")
        extractor = RenpyExtractor(game_root)
        analysis = extractor.analyze_and_extract()
        units = analysis['units']
        
        if not units:
            raise ValueError('Nenhuma unidade traduzivel encontrada nos scripts do jogo.')
        
        # 4. Fine-Tuning LoRA Opcional (Adaptacao ao Jogo)
        print(f"\n=== ETAPA 4: TREINAMENTO LoRA (ADAPTACAO AO JOGO) ===")
        trainer = RenplayTrainer(model, tokenizer, checkpoint_dir=DRIVE_DIR / 'checkpoints')
        training_samples = trainer.prepare_dataset(units, target_lang='pt-BR')
        
        # Executa fine-tuning LoRA rapido de 60 passos no contexto do jogo
        if len(training_samples) > 20:
            trainer.train_lora(training_samples[:300], max_steps=60, output_name=f"lora_{item['name']}")
        else:
            FastLanguageModel.for_inference(model)
        
        # 5. Traducao em Lote com Barra de Progresso
        print(f"\n=== ETAPA 5: TRADUCAO COM PROTECAO DE TAGS ===")
        queue_mgr.update_status(item['id'], 'translating')
        translations = []
        target_lang = item['target_languages'][0] if item['target_languages'] else 'pt-BR'
        
        for unit in tqdm(units, desc='Traduzindo falas', unit='linha'):
            orig = unit['original_text']
            char = unit.get('character', 'Narrador')
            
            # 1. Consulta a memoria SQLite antes de acionar a GPU
            cached = memory.get_translation(orig, target_lang=target_lang, character=char)
            if cached:
                translations.append({'original_text': orig, 'translated_text': cached})
                continue
            
            # 2. Mascarar tags e variaveis
            masked_text, mask_map = protector.protect(orig)
            
            # 3. Geracao com Qwen3.5-4B
            prompt = f"<|im_start|>system\nVoce e a Renplay AI. Traduza o dialogo para Portugues do Brasil preservando todas as mascaras __TAG_X__ e __VAR_X__ intactas.<|im_end|>\n<|im_start|>user\nPersonagem: {char}\nTexto: {masked_text}<|im_end|>\n<|im_start|>assistant\n"
            inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
            outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.3, use_cache=True)
            raw_resp = tokenizer.batch_decode(outputs)[0]
            pred = raw_resp.split('<|im_start|>assistant\n')[-1].replace('<|im_end|>', '').strip()
            
            # 4. Restaurar tags e variaveis originais
            final_text = protector.restore(pred, mask_map)
            
            # 5. Salvar na memoria SQLite para reaproveitamento permanente
            memory.save_translation(orig, final_text, target_lang=target_lang, character=char, game_name=item['name'])
            translations.append({'original_text': orig, 'translated_text': final_text})
        
        # 6. Gerar Patch de Arquivo Unico com Seletor de Idiomas
        print(f"\n=== ETAPA 6: EMPACOTAMENTO DO PATCH DE ARQUIVO UNICO ===")
        patch_file = patch_builder.build_single_file_patch(
            game_name=item['name'],
            translations=translations,
            target_lang=target_lang,
        )
        
        queue_mgr.update_status(item['id'], 'completed', patch_path=str(patch_file))
        print(f"\n" + "=" * 65)
        print(f"[SUCESSO COMPLETO!] Patch gerado e salvo no Google Drive:")
        print(f"{patch_file}")
        print(f"Total de falas traduzidas: {len(translations)}")
        print("Instrucao: Basta copiar este unico arquivo para a pasta game/ do jogo!")
        print("=" * 65 + "\n")
        
    except Exception as e:
        print(f"[ERRO] Falha no pipeline: {e}")
        queue_mgr.update_status(item['id'], 'error', error_message=str(e))
